# NYC Taxi Databricks Analytics

End-to-end lakehouse workflow for NYC green and yellow taxi trip data, covering ingestion, cleaning, borough enrichment, business analytics, and machine learning in Databricks.


## 1. Lakehouse Ingestion and Data Quality


### 1.1 Environment and Storage Setup


In [0]:
# Configure the Spark session for NYC-local temporal features.
spark.conf.set("spark.sql.session.timeZone", "America/New_York")


In [0]:
# Core workspace configuration. Update these values if your Databricks catalog layout differs.
CATALOG = "workspace"
SCHEMA = "bde"
VOLUME = "nyc_taxi"

RAW_GREEN_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/green"
RAW_YELLOW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/yellow"
TAXI_ZONE_LOOKUP_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/taxi_zone_lookup.csv"

TAXI_ZONE_TABLE = f"{CATALOG}.{SCHEMA}.taxi_zone_lookup"
TRIPS_TABLE = f"{CATALOG}.{SCHEMA}.taxi_trips_cleaned_borough"
MODEL_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/models/model_a_ridge_v1"

# Create the lakehouse objects and folders required by the notebook.
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

dbutils.fs.mkdirs(RAW_GREEN_PATH)
dbutils.fs.mkdirs(RAW_YELLOW_PATH)


In [0]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("gdown") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown"])

from gdown import download


def fetch_to_volume(file_id: str, destination_dir: str, filename: str) -> str:
    """Download a Google Drive file into a Databricks volume path."""
    destination_path = f"{destination_dir}/{filename}"
    url = f"https://drive.google.com/uc?id={file_id}"
    download(url, destination_path, quiet=False)
    return destination_path


In [0]:
GREEN_TAXI_FILES = [
    {
        "file_id": "1T7o40ZqmAK90W5o9TBOPrlpECBZ9KedV",
        "filename": "green_taxi.parquet",
    }
] 

for source in GREEN_TAXI_FILES:
    fetch_to_volume(source["file_id"], RAW_GREEN_PATH, source["filename"])


In [0]:
YELLOW_TAXI_FILES = [
    {
        "file_id": "1Iayo8ZaL4oOB2v65VpnumdDSOVz3xoNf",
        "filename": "yellow_taxi.parquet",
    }
] 

for source in YELLOW_TAXI_FILES:
    fetch_to_volume(source["file_id"], RAW_YELLOW_PATH, source["filename"])


In [0]:
print("Raw green taxi files")
display(dbutils.fs.ls(RAW_GREEN_PATH))

print("Raw yellow taxi files")
display(dbutils.fs.ls(RAW_YELLOW_PATH))


In [0]:
green_df = spark.read.parquet(RAW_GREEN_PATH)
yellow_df = spark.read.parquet(RAW_YELLOW_PATH)

raw_counts = {
    "green": green_df.count(),
    "yellow": yellow_df.count(),
}
raw_total_count = sum(raw_counts.values())

print(f"Green taxi rows: {raw_counts['green']:,}")
print(f"Yellow taxi rows: {raw_counts['yellow']:,}")
print(f"Raw total rows: {raw_total_count:,}")


In [0]:
display(green_df)
display(yellow_df)

In [0]:
print("Green taxi schema")
green_df.printSchema()
display(green_df.limit(5))

print("Yellow taxi schema")
yellow_df.printSchema()
display(yellow_df.limit(5))


### 1.2 Taxi Zone Lookup to Delta


In [0]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

taxi_zone_schema = StructType([
    StructField("LocationID", IntegerType(), False),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True),
])

taxi_zone_df = (
    spark.read
    .schema(taxi_zone_schema)
    .option("header", True)
    .csv(TAXI_ZONE_LOOKUP_PATH)
)

location_count = taxi_zone_df.count()
distinct_location_count = taxi_zone_df.select("LocationID").distinct().count()
assert distinct_location_count == location_count, "Duplicate LocationID values found in taxi_zone_lookup"

taxi_zone_df.write.mode("overwrite").saveAsTable(TAXI_ZONE_TABLE)
print(f"Taxi zones loaded: {spark.table(TAXI_ZONE_TABLE).count():,}")


In [0]:
%sql
SELECT *
FROM taxi_zone_lookup;


### 1.3 Schema Harmonization and Union


In [0]:
from pyspark.sql import functions as F

# Standardize yellow taxi columns to the shared trip schema.
yellow_std = (
    yellow_df
    .withColumn("color", F.lit("yellow"))
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
    .withColumn("payment_type", F.col("payment_type").cast("int"))
    .withColumn("RatecodeID", F.col("RatecodeID").cast("int"))
    .withColumn("passenger_count", F.col("passenger_count").cast("int"))
)

# Standardize green taxi columns to the shared trip schema.
green_std = (
    green_df
    .withColumn("color", F.lit("green"))
    .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime")
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
    .withColumn("payment_type", F.col("payment_type").cast("int"))
    .withColumn("RatecodeID", F.col("RatecodeID").cast("int"))
    .withColumn("trip_type", F.col("trip_type").cast("int"))
    .withColumn("passenger_count", F.col("passenger_count").cast("int"))
)

trips_raw = yellow_std.unionByName(green_std, allowMissingColumns=True)
trips_raw.createOrReplaceTempView("trips_raw")


In [0]:
# trips_raw overview:
print(f"trips_raw.count() = {trips_raw.count():,}")
trips_raw.printSchema()
trips_raw.show(5, truncate=False)

### 1.4 Trip Feature Engineering


In [0]:
%sql
-- Derive normalized timestamps and trip-level distance/speed features.
CREATE OR REPLACE TEMP VIEW trips_enriched AS
SELECT
  *,
  CAST(pickup_datetime AS TIMESTAMP) AS pickup_ts,
  CAST(dropoff_datetime AS TIMESTAMP) AS dropoff_ts,
  (CAST(dropoff_datetime AS BIGINT) - CAST(pickup_datetime AS BIGINT)) / 60.0 AS duration_min,
  trip_distance * 1.60934 AS trip_distance_km,
  try_divide(
    trip_distance * 1.60934,
    NULLIF((CAST(dropoff_datetime AS BIGINT) - CAST(pickup_datetime AS BIGINT)) / 3600.0, 0)
  ) AS speed_kmh
FROM trips_raw;


### 1.5 Data Quality Rules and Row-Retention Controls


In [0]:
%sql
-- Inspect observed time bounds
WITH b AS (
  SELECT MIN(pickup_ts) AS min_pickup, MAX(dropoff_ts) AS max_dropoff
  FROM trips_enriched
)
SELECT * FROM b;


In [0]:
%sql
-- Apply conservative quality filters while preserving nullable business fields.
CREATE OR REPLACE TEMP VIEW trips_clean AS
SELECT *
FROM trips_enriched
WHERE
  dropoff_ts > pickup_ts
  AND duration_min BETWEEN 1 AND 180
  AND trip_distance_km BETWEEN 0.1 AND 200
  AND speed_kmh > 0 AND speed_kmh <= 120
  AND (passenger_count IS NULL OR passenger_count BETWEEN 0 AND 6)
  AND total_amount >= 0
  AND (PULocationID BETWEEN 1 AND 265 OR PULocationID IS NULL)
  AND (DOLocationID BETWEEN 1 AND 265 OR DOLocationID IS NULL)
  AND (
        (color = 'yellow' AND pickup_ts >= TIMESTAMP('2009-01-01') AND dropoff_ts < TIMESTAMP('2025-01-01'))
     OR (color = 'green'  AND pickup_ts >= TIMESTAMP('2013-08-01') AND dropoff_ts < TIMESTAMP('2025-01-01'))
      )
  AND (RatecodeID IS NULL OR CAST(RatecodeID AS INT) IN (1, 2, 3, 4, 5, 6, 99))
  AND (payment_type IS NULL OR CAST(payment_type AS INT) IN (0, 1, 2, 3, 4, 5, 6))
  AND (trip_type IS NULL OR CAST(trip_type AS INT) IN (1, 2))
  AND (store_and_fwd_flag IS NULL OR store_and_fwd_flag IN ('Y', 'N'));


In [0]:
%sql
-- --- Removal audit (ensure <= 10%) ---
WITH base AS (SELECT COUNT(*) AS n FROM trips_enriched),
     clean AS (SELECT COUNT(*) AS n FROM trips_clean)
SELECT base.n AS raw_rows, clean.n AS clean_rows,
       (base.n - clean.n) AS removed_rows,
       ROUND((base.n - clean.n)/base.n*100, 2) AS removed_pct
FROM base, clean;


### 1.6 Borough Enrichment and Curated Table Persistence


In [0]:
%sql
-- Enrich pickup and drop-off locations with taxi zone and borough attributes.
CREATE OR REPLACE TEMP VIEW trips_enriched_borough AS
SELECT
  t.*,
  COALESCE(pu.Borough, 'Unknown') AS pu_borough,
  COALESCE(do.Borough, 'Unknown') AS do_borough,
  COALESCE(pu.Zone, 'Unknown') AS pu_zone,
  COALESCE(do.Zone, 'Unknown') AS do_zone
FROM trips_clean t
LEFT JOIN taxi_zone_lookup pu
  ON t.PULocationID = pu.LocationID
LEFT JOIN taxi_zone_lookup do
  ON t.DOLocationID = do.LocationID;

CREATE OR REPLACE TEMP VIEW trips_final AS
SELECT
  e.*,
  year(pickup_ts) AS year,
  month(pickup_ts) AS month
FROM trips_enriched_borough e;

DROP TABLE IF EXISTS taxi_trips_cleaned_borough;

CREATE TABLE taxi_trips_cleaned_borough
USING DELTA
PARTITIONED BY (year, month)
AS SELECT * FROM trips_final;

SELECT COUNT(*) AS final_rows
FROM taxi_trips_cleaned_borough;


## 2. Business Analytics


### 2.1 Monthly Trip and Revenue Summary


In [0]:
%sql
-- Monthly operating summary with peak weekday and peak hour context.
WITH base AS (
  SELECT
    year,
    month,
    make_date(year, month, 1) AS ym,
    date_format(pickup_ts, 'EEEE') AS dow_name,
    hour(pickup_ts) AS hr,
    passenger_count,
    total_amount
  FROM taxi_trips_cleaned_borough
),
agg AS (
  SELECT
    ym,
    COUNT(*) AS total_trips,
    AVG(passenger_count) AS avg_passengers,
    AVG(total_amount) AS avg_amount_per_trip,
    AVG(CASE WHEN passenger_count > 0 THEN try_divide(total_amount, passenger_count) END) AS avg_amount_per_passenger
  FROM base
  GROUP BY ym
),
dow_rank AS (
  SELECT
    ym,
    dow_name,
    COUNT(*) AS trips_by_dow,
    ROW_NUMBER() OVER (PARTITION BY ym ORDER BY COUNT(*) DESC, dow_name) AS rn
  FROM base
  GROUP BY ym, dow_name
),
hr_rank AS (
  SELECT
    ym,
    hr,
    COUNT(*) AS trips_by_hr,
    ROW_NUMBER() OVER (PARTITION BY ym ORDER BY COUNT(*) DESC, hr) AS rn
  FROM base
  GROUP BY ym, hr
)
SELECT
  a.ym,
  a.total_trips,
  d.dow_name AS most_trips_dow,
  h.hr AS most_trips_hour,
  ROUND(a.avg_passengers, 2) AS avg_passengers,
  ROUND(a.avg_amount_per_trip, 2) AS avg_amount_per_trip,
  ROUND(a.avg_amount_per_passenger, 2) AS avg_amount_per_passenger
FROM agg a
LEFT JOIN dow_rank d ON a.ym = d.ym AND d.rn = 1
LEFT JOIN hr_rank h ON a.ym = h.ym AND h.rn = 1
ORDER BY a.ym;


### 2.2 Descriptive Statistics by Taxi Color


In [0]:
%sql
-- Distribution profile by taxi service color.
WITH base AS (
  SELECT color, duration_min, trip_distance_km, speed_kmh
  FROM taxi_trips_cleaned_borough
  WHERE duration_min IS NOT NULL
    AND trip_distance_km IS NOT NULL
    AND speed_kmh IS NOT NULL
)
SELECT
  color,
  ROUND(AVG(duration_min), 2) AS duration_avg_min,
  ROUND(percentile_approx(duration_min, 0.5, 10000), 2) AS duration_med_min,
  ROUND(MIN(duration_min), 2) AS duration_min_min,
  ROUND(MAX(duration_min), 2) AS duration_max_min,
  ROUND(AVG(trip_distance_km), 2) AS dist_avg_km,
  ROUND(percentile_approx(trip_distance_km, 0.5, 10000), 2) AS dist_med_km,
  ROUND(MIN(trip_distance_km), 2) AS dist_min_km,
  ROUND(MAX(trip_distance_km), 2) AS dist_max_km,
  ROUND(AVG(speed_kmh), 2) AS speed_avg_kmh,
  ROUND(percentile_approx(speed_kmh, 0.5, 10000), 2) AS speed_med_kmh,
  ROUND(MIN(speed_kmh), 2) AS speed_min_kmh,
  ROUND(MAX(speed_kmh), 2) AS speed_max_kmh
FROM base
GROUP BY color
ORDER BY color;


### 2.3 Route Demand Grid by Color, Borough Pair, Month, Weekday, and Hour


In [0]:
%sql
-- Route demand and revenue grid for downstream operational analysis.
SELECT
  color,
  pu_borough,
  do_borough,
  make_date(year, month, 1) AS ym,
  date_format(pickup_ts, 'EEEE') AS dow_name,
  hour(pickup_ts) AS hr,
  COUNT(*) AS total_trips,
  ROUND(AVG(trip_distance_km), 2) AS avg_distance_km,
  ROUND(AVG(total_amount), 2) AS avg_amount_per_trip,
  ROUND(SUM(total_amount), 2) AS total_amount
FROM taxi_trips_cleaned_borough
GROUP BY color, pu_borough, do_borough, year, month, date_format(pickup_ts, 'EEEE'), hour(pickup_ts)
ORDER BY ym, color, pu_borough, do_borough, dow_name, hr;


### 2.4 Top 2024 Pickup-to-Dropoff Revenue Share


In [0]:
%sql
-- Highest-revenue borough-pair routes and share of 2024 revenue.
WITH y2024 AS (
  SELECT *
  FROM taxi_trips_cleaned_borough
  WHERE year = 2024
),
pair_rev AS (
  SELECT pu_borough, do_borough, SUM(total_amount) AS pair_total
  FROM y2024
  GROUP BY pu_borough, do_borough
),
overall AS (
  SELECT SUM(pair_total) AS overall_total FROM pair_rev
),
top10 AS (
  SELECT
    pu_borough,
    do_borough,
    pair_total,
    ROW_NUMBER() OVER (ORDER BY pair_total DESC) AS rn
  FROM pair_rev
  ORDER BY pair_total DESC
  LIMIT 10
)
SELECT
  t.pu_borough,
  t.do_borough,
  ROUND(t.pair_total, 2) AS pair_total_amount_2024,
  ROUND(t.pair_total / o.overall_total * 100, 2) AS share_pct_2024
FROM top10 t
CROSS JOIN overall o
ORDER BY t.pair_total DESC;


### 2.5 Tip Percentage Analysis


In [0]:
%sql
-- Tip participation and high-tip share among tipped trips.
WITH nums AS (
  SELECT
    COUNT(*) AS n_all,
    SUM(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END) AS n_tipped,
    SUM(CASE WHEN tip_amount >= 15 THEN 1 ELSE 0 END) AS n_tip_15_plus
  FROM taxi_trips_cleaned_borough
)
SELECT
  ROUND((n_tipped * 100.0) / NULLIF(n_all, 0), 2) AS pct_with_tips,
  ROUND((n_tip_15_plus * 100.0) / NULLIF(n_tipped, 0), 2) AS pct_tips_15_plus_among_tipped
FROM nums;


### 2.6 Duration Bands, Average Speed, and Distance Efficiency


In [0]:
%sql
-- Speed and distance efficiency by trip-duration band.
WITH binned AS (
  SELECT
    CASE
      WHEN duration_min < 5 THEN 'Under 5 mins'
      WHEN duration_min >= 5 AND duration_min < 10 THEN '5-10 mins'
      WHEN duration_min >= 10 AND duration_min < 20 THEN '10-20 mins'
      WHEN duration_min >= 20 AND duration_min < 30 THEN '20-30 mins'
      WHEN duration_min >= 30 AND duration_min < 60 THEN '30-60 mins'
      ELSE '60+ mins'
    END AS duration_bin,
    speed_kmh,
    trip_distance_km,
    total_amount
  FROM taxi_trips_cleaned_borough
)
SELECT
  duration_bin,
  ROUND(AVG(speed_kmh), 2) AS avg_speed_kmh,
  ROUND(AVG(CASE WHEN total_amount > 0 THEN trip_distance_km / total_amount END), 4) AS avg_km_per_dollar
FROM binned
GROUP BY duration_bin
ORDER BY
  CASE duration_bin
    WHEN 'Under 5 mins' THEN 1
    WHEN '5-10 mins' THEN 2
    WHEN '10-20 mins' THEN 3
    WHEN '20-30 mins' THEN 4
    WHEN '30-60 mins' THEN 5
    ELSE 6
  END;


## 3. Machine Learning


### 3.1 Imports and Reusable Helpers


In [0]:
# Machine learning helpers
from pyspark.sql import functions as F
from pyspark.sql import types as T
import numpy as np

spark.conf.set("spark.sql.session.timeZone", "America/New_York")
TABLE = TRIPS_TABLE


def predict_linear(df, feature_cols, weights, output_col):
    """Score a Spark DataFrame with a linear model represented by a NumPy weight vector."""
    weight_values = [float(value) for value in weights.tolist()]
    prediction = None
    for weight, feature_col in zip(weight_values, feature_cols):
        term = F.lit(weight) * F.col(feature_col)
        prediction = term if prediction is None else prediction + term
    return df.withColumn(output_col, prediction)


def rmse(df, prediction_col, label_col):
    """Compute root mean squared error for Spark columns."""
    return float(
        df.select(F.pow(F.col(prediction_col) - F.col(label_col), 2).alias("squared_error"))
        .agg(F.sqrt(F.avg("squared_error")).alias("rmse"))
        .first()["rmse"]
    )


### 3.2 Train, Validation, and Test Splits


In [0]:
# Time-based modeling splits and base feature columns
base = (
    spark.table(TABLE)
    .select(
        "total_amount",
        "trip_distance_km",
        "duration_min",
        F.coalesce(F.col("passenger_count"), F.lit(1)).alias("passenger_count"),
        "color",
        "pu_borough",
        "do_borough",
        "year",
        "month",
        F.dayofweek("pickup_ts").alias("dow"),
        F.hour("pickup_ts").alias("hour"),
        "pickup_ts",
    )
    .where("total_amount is not null and duration_min is not null and trip_distance_km is not null")
)

train = base.where("pickup_ts < timestamp('2024-09-01')")
val = base.where("pickup_ts >= timestamp('2024-09-01') and pickup_ts < timestamp('2024-10-01')")
test = base.where("pickup_ts >= timestamp('2024-10-01') and pickup_ts < timestamp('2025-01-01')")

n_train = train.count()
n_val = val.count()
n_test = test.count()
print(f"Rows - train: {n_train:,} | val: {n_val:,} | test (Oct-Dec 2024): {n_test:,}")


### 3.3 Hierarchical Baseline Model

- Estimate average `total_amount` by route, taxi color, month, weekday, and hour.
- Back off to coarser groupings and finally the global mean to avoid missing predictions.
- Evaluate RMSE on validation and test datasets.


In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import broadcast

# Hierarchical baseline model using increasingly general route-time averages.
KEY_LEVELS = [
    ["color_k", "pu_k", "do_k", "month", "dow", "hour"],
    ["color_k", "pu_k", "do_k", "month", "dow"],
    ["color_k", "pu_k", "do_k", "month"],
    ["color_k", "month"],
]


def add_key_norm_cols(df):
    """Normalize nullable categorical join keys used by the baseline maps."""
    return (
        df
        .withColumn("color_k", F.coalesce(F.col("color"), F.lit("Unknown")))
        .withColumn("pu_k", F.coalesce(F.col("pu_borough"), F.lit("Unknown")))
        .withColumn("do_k", F.coalesce(F.col("do_borough"), F.lit("Unknown")))
    )


trainval = add_key_norm_cols(train.unionByName(val))

baseline_maps = []
for keys in KEY_LEVELS:
    mapping = trainval.groupBy(*keys).agg(F.avg("total_amount").alias("pred"))
    baseline_maps.append((keys, broadcast(mapping)))

global_mean = float(trainval.agg(F.avg("total_amount")).first()[0])


def apply_baseline(df_in: DataFrame) -> DataFrame:
    """Apply hierarchical fallback averages from most specific keys to global mean."""
    df = add_key_norm_cols(df_in)
    prediction_cols = []
    for i, (keys, mapping) in enumerate(baseline_maps):
        pred_col = f"pred_l{i}"
        df = df.join(mapping.withColumnRenamed("pred", pred_col), on=keys, how="left")
        prediction_cols.append(pred_col)
    return df.withColumn(
        "prediction_baseline",
        F.coalesce(*[F.col(col_name) for col_name in prediction_cols], F.lit(global_mean)),
    )


val_b = apply_baseline(val)
test_b = apply_baseline(test)

rmse_val_baseline = rmse(val_b, prediction_col="prediction_baseline", label_col="total_amount")
rmse_test_baseline = rmse(test_b, prediction_col="prediction_baseline", label_col="total_amount")

print(f"Baseline RMSE - val: {rmse_val_baseline:.3f} | test: {rmse_test_baseline:.3f}")


In [0]:
# Distribution sanity check for total_amount
for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"\n{name.upper()}")
    df.selectExpr(
        "AVG(total_amount) as mean_amt",
        "STDDEV_POP(total_amount) as std_amt",
        "percentile_approx(total_amount, 0.5, 1000) as p50",
        "percentile_approx(total_amount, 0.9, 1000) as p90",
        "percentile_approx(total_amount, 0.99, 1000) as p99",
        "MAX(total_amount) as max_amt"
    ).show(truncate=False)

In [0]:
# Robust label clipping (winsorization)

# Compute a high percentile CAP from TRAIN ONLY (no leakage)
cap_row = (train
           .selectExpr("percentile_approx(total_amount, 0.9995, 10000) as p9995")  # 99.95th
           .first())
CAP = float(cap_row["p9995"]) if cap_row and cap_row["p9995"] is not None else 500.0

print(f"Using label cap (train 99.95th): {CAP:.2f}")

def clip_amount(col, cap):
    return F.when(col > F.lit(cap), F.lit(cap)).otherwise(col)

# We'll add a ROBUST label to each split (same CAP applied to val/test for 'robust RMSE' reporting)
train = train.withColumn("label_robust", clip_amount(F.col("total_amount"), CAP))
val   = val  .withColumn("label_robust", clip_amount(F.col("total_amount"), CAP))
test  = test .withColumn("label_robust", clip_amount(F.col("total_amount"), CAP))

### 3.4 Target Encoding

Target encoding is fitted on the training set only, then joined into validation and test datasets with missing values filled from the training global mean.


In [0]:
# Smoothed target encoding fitted on training data only.
TE_COLS = ["color", "pu_borough", "do_borough"]
y_global = float(train.agg(F.avg("label_robust")).first()[0])


def fit_te_smoothed(df_train, col_name, smoothing=30.0):
    encoded = (
        df_train
        .groupBy(col_name)
        .agg(
            F.sum("label_robust").alias("sum_y"),
            F.count(F.lit(1)).alias("cnt"),
        )
    )
    return encoded.select(
        F.col(col_name),
        (
            (F.col("sum_y") + F.lit(smoothing) * F.lit(y_global))
            / (F.col("cnt") + F.lit(smoothing))
        ).alias(f"{col_name}_te"),
    )


te_smoothing = 30.0
te_maps = {col_name: fit_te_smoothed(train, col_name, smoothing=te_smoothing) for col_name in TE_COLS}


def apply_target_encoding(df_in):
    df = df_in
    for col_name in TE_COLS:
        encoded_col = f"{col_name}_te"
        df = df.join(te_maps[col_name], on=col_name, how="left")
        df = df.withColumn(encoded_col, F.coalesce(F.col(encoded_col), F.lit(y_global)))
    return df


train_te = apply_target_encoding(train)
val_te = apply_target_encoding(val)
test_te = apply_target_encoding(test)

train_te = train_te.withColumnRenamed("pu_borough_te", "pu_te").withColumnRenamed("do_borough_te", "do_te")
val_te = val_te.withColumnRenamed("pu_borough_te", "pu_te").withColumnRenamed("do_borough_te", "do_te")
test_te = test_te.withColumnRenamed("pu_borough_te", "pu_te").withColumnRenamed("do_borough_te", "do_te")


### 3.5 Feature Standardization


In [0]:
# Feature standardization fitted on training data only.
NUM_COLS = [
    "trip_distance_km",
    "duration_min",
    "passenger_count",
    "hour",
    "month",
    "dow",
    "color_te",
    "pu_te",
    "do_te",
]

stats = (
    train_te
    .agg(
        *[F.avg(col_name).alias(f"{col_name}_mean") for col_name in NUM_COLS],
        *[F.stddev_pop(col_name).alias(f"{col_name}_std") for col_name in NUM_COLS],
    )
    .collect()[0]
    .asDict()
)


def apply_zscore(df):
    out = df
    for col_name in NUM_COLS:
        mean = stats[f"{col_name}_mean"]
        std = stats[f"{col_name}_std"] if stats[f"{col_name}_std"] and stats[f"{col_name}_std"] > 0 else 1.0
        out = out.withColumn(f"{col_name}_z", (F.col(col_name) - F.lit(mean)) / F.lit(std))
    return out


train_z = apply_zscore(train_te)
val_z = apply_zscore(val_te)
test_z = apply_zscore(test_te)


### 3.6 Feature Assembly


In [0]:
# Assemble model-ready feature columns.
FEATURE_EXPRESSIONS = [
    ("bias", F.lit(1.0)),
    ("dist", F.col("trip_distance_km_z")),
    ("dur", F.col("duration_min_z")),
    ("pc", F.col("passenger_count_z")),
    ("hr", F.col("hour_z")),
    ("mo", F.col("month_z")),
    ("dw", F.col("dow_z")),
    ("color_te", F.col("color_te_z")),
    ("pu_te", F.col("pu_te_z")),
    ("do_te", F.col("do_te_z")),
]
FEATURE_COLS = [name for name, _ in FEATURE_EXPRESSIONS]
FEAT_COLS = FEATURE_COLS  # Backward-compatible alias for downstream model cells.


def with_features(df):
    out = df
    for name, expr in FEATURE_EXPRESSIONS:
        out = out.withColumn(name, expr.cast("double"))
    return out.select(*FEATURE_COLS, "total_amount", "label_robust")


train_f = with_features(train_z)
val_f = with_features(val_z)
test_f = with_features(test_z)

print("Feature dimension:", len(FEATURE_COLS))


### 3.7 Model Training and Evaluation


In [0]:
# Model A: closed-form ridge regression

# Hyperparameters
lam = 1e-3
p = len(FEAT_COLS)

# Build X'X and X'y where y = label_robust
agg_exprs = []
for i,ci in enumerate(FEAT_COLS):
    for j,cj in enumerate(FEAT_COLS):
        if j >= i:
            agg_exprs.append(F.sum(F.col(ci)*F.col(cj)).alias(f"G_{i}_{j}"))
b_exprs = [F.sum(F.col(ci)*F.col("label_robust")).alias(f"b_{i}") for i,ci in enumerate(FEAT_COLS)]
row = train_f.agg(*(agg_exprs + b_exprs)).collect()[0].asDict()

G = np.zeros((p,p), dtype=float)
for i in range(p):
    for j in range(i,p):
        G[i,j] = float(row[f"G_{i}_{j}"]); G[j,i] = G[i,j]
b = np.array([float(row[f"b_{i}"]) for i in range(p)], dtype=float)

w_A = np.linalg.solve(G + lam*np.eye(p), b)

# Score validation and test datasets
val_A  = predict_linear(val_f, FEAT_COLS, w_A, out_col="prediction_A")
test_A = predict_linear(test_f, FEAT_COLS, w_A, out_col="prediction_A")

rmse_val_A_true   = rmse(val_A,  "prediction_A", "total_amount")
rmse_test_A_true  = rmse(test_A, "prediction_A", "total_amount")
rmse_val_A_robust  = rmse(val_A,  "prediction_A", "label_robust")
rmse_test_A_robust = rmse(test_A, "prediction_A", "label_robust")

print(f"Model A RMSE (TRUE)   — val: {rmse_val_A_true:.3f} | test: {rmse_test_A_true:.3f}")
print(f"Model A RMSE (ROBUST) — val: {rmse_val_A_robust:.3f} | test: {rmse_test_A_robust:.3f}")

In [0]:
# Model B: ridge regression with gradient descent

# Hyperparameters
gd_lr = 0.1
gd_epochs = 5
lam_gd = 1e-3
gd_sample_frac = 0.05  # speed knob; set 1.0 for all rows

# Build GD mini-batch with features + explicit label column
# NOTE: we carry the true label ('label' = total_amount), and we'll clip in the loop.

train_gd = train_f.select(*FEAT_COLS, F.col("total_amount").alias("label"))
if gd_sample_frac < 1.0:
    train_gd = train_gd.sample(False, gd_sample_frac, seed=42)

# Init weights
w_B = np.zeros(len(FEAT_COLS), dtype=float)

for epoch in range(gd_epochs):
    # Build pred column on the DF to avoid unresolved alias issues
    pred_col = None
    for i, c in enumerate(FEAT_COLS):
        term = F.lit(float(w_B[i])) * F.col(c)
        pred_col = term if pred_col is None else (pred_col + term)

    # Residual to ROBUST label: least(label, CAP)
    gd_with_pred = (train_gd
                    .withColumn("pred", pred_col)
                    .withColumn("resid", F.col("pred") - F.least(F.col("label"), F.lit(CAP))))

    # Gradients: 2/n * sum(x_i * resid)  +  2*lambda*w (bias unpenalized)
    agg_exprs = [F.sum(F.col(c) * F.col("resid")).alias(f"g_{i}") for i, c in enumerate(FEAT_COLS)]
    agg_exprs.append(F.count(F.lit(1)).alias("n"))
    row = gd_with_pred.agg(*agg_exprs).collect()[0].asDict()

    n = float(row["n"]) if row["n"] else 1.0
    g = np.array([float(row[f"g_{i}"]) for i in range(len(FEAT_COLS))], dtype=float) * (2.0 / n)

    reg = lam_gd * w_B
    reg[0] = 0.0 
    g += 2.0 * reg

    w_B = w_B - gd_lr * g
    print(f"Epoch {epoch + 1}/{gd_epochs}: |w|={np.linalg.norm(w_B):.3f}")

# Score validation and test datasets
val_B  = predict_linear(val_f, FEAT_COLS, w_B, out_col="prediction_B")
test_B = predict_linear(test_f, FEAT_COLS, w_B, out_col="prediction_B")

rmse_val_B_true   = rmse(val_B,  "prediction_B", "total_amount")
rmse_test_B_true  = rmse(test_B, "prediction_B", "total_amount")
rmse_val_B_robust  = rmse(val_B,  "prediction_B", "label_robust")
rmse_test_B_robust = rmse(test_B, "prediction_B", "label_robust")

print(f"Model B RMSE (TRUE)   — val: {rmse_val_B_true:.3f} | test: {rmse_test_B_true:.3f}")
print(f"Model B RMSE (ROBUST) — val: {rmse_val_B_robust:.3f} | test: {rmse_test_B_robust:.3f}")

In [0]:
# Model C: robust Huber regression

# Hyperparameters
delta = 20.0        # Huber threshold in dollars; 10–30 works well on fares
lam   = 1e-3        # L2 weight decay; no penalty on bias
lr    = 0.05        # learning rate
epochs = 10
sample_frac = 0.1   # use 1.0 for full data (slower)

train_h = train_f.select(*FEAT_COLS, F.col("total_amount").alias("label"))
if sample_frac < 1.0:
    train_h = train_h.sample(False, sample_frac, seed=42)

# Initialize weights
w_C = np.zeros(len(FEAT_COLS), dtype=float)

for ep in range(epochs):
    # pred = Xw
    pred = None
    for i, c in enumerate(FEAT_COLS):
        t = F.lit(float(w_C[i])) * F.col(c)
        pred = t if pred is None else (pred + t)

    # Huber residuals: psi(r) = r if |r|<=delta else delta*sign(r)
    r = (pred - F.col("label"))
    psi = F.when(F.abs(r) <= F.lit(delta), r) \
           .otherwise(F.lit(delta) * F.signum(r))

    # Add prediction and psi columns to DataFrame
    train_h_pred = train_h.withColumn("pred", pred).withColumn("psi", psi)

    # Aggregate gradient: (1/n) * X^T psi + lam * w   (bias unpenalized)
    grads = [F.sum(F.col(c) * F.col("psi")).alias(f"g_{i}") for i, c in enumerate(FEAT_COLS)]
    aggs = train_h_pred.agg(*grads, F.count(F.lit(1)).alias("n")).collect()[0].asDict()
    n = float(aggs["n"]) if aggs["n"] else 1.0
    g = np.array([float(aggs[f"g_{i}"]) for i in range(len(FEAT_COLS))], dtype=float) / max(n, 1.0)

    reg = lam * w_C
    reg[0] = 0.0
    g += reg

    w_C = w_C - lr * g
    print(f"[Huber] Epoch {ep + 1}/{epochs} |w|={np.linalg.norm(w_C):.3f}")

# Score validation and test datasets
val_C  = predict_linear(val_f, FEAT_COLS, w_C, out_col="prediction_C")
test_C = predict_linear(test_f, FEAT_COLS, w_C, out_col="prediction_C")

rmse_val_C_true   = rmse(val_C,  "prediction_C", "total_amount")
rmse_test_C_true  = rmse(test_C, "prediction_C", "total_amount")
rmse_val_C_robust  = rmse(val_C,  "prediction_C", "label_robust")
rmse_test_C_robust = rmse(test_C, "prediction_C", "label_robust")

print(f"Model C (Huber) RMSE (TRUE)   — val: {rmse_val_C_true:.3f} | test: {rmse_test_C_true:.3f}")
print(f"Model C (Huber) RMSE (ROBUST) — val: {rmse_val_C_robust:.3f} | test: {rmse_test_C_robust:.3f}")

In [0]:
# Model D: fallback histogram tree

# Hyperparameters
BFEATS = ["dist","dur","pu_te","do_te"]
quantiles = [0.2, 0.4, 0.6, 0.8]  # 5 bins per feature

# Compute thresholds from TRAIN
exprs = [
    F.expr(f"percentile_approx({c}, array({','.join(str(q) for q in quantiles)}), 10000)").alias(f"q_{c}")
    for c in BFEATS
]
qs = train_f.agg(*exprs).first().asDict()

def bin_expr(colname, qvals):
    # qvals is a Python list of 4 thresholds => 5 bins [0..4]
    return (F.when(F.col(colname) <= F.lit(qvals[0]), F.lit(0))
            .when(F.col(colname) <= F.lit(qvals[1]), F.lit(1))
            .when(F.col(colname) <= F.lit(qvals[2]), F.lit(2))
            .when(F.col(colname) <= F.lit(qvals[3]), F.lit(3))
            .otherwise(F.lit(4))).cast("int")

# TRAIN leaves
train_bins = train_f
for c in BFEATS:
    train_bins = train_bins.withColumn(f"{c}_bin", bin_expr(c, qs[f"q_{c}"]))

leaves = (train_bins
          .groupBy(*[f"{c}_bin" for c in BFEATS])
          .agg(F.avg("total_amount").alias("leaf_pred"),
               F.count("*").alias("n_leaf")))

# Apply to VAL/TEST
def apply_bins(df):
    out = df
    for c in BFEATS:
        out = out.withColumn(f"{c}_bin", bin_expr(c, qs[f"q_{c}"]))
    return out

val_bins   = apply_bins(val_f)
test_bins  = apply_bins(test_f)

# Join leaf means; fallback = global train mean if unseen leaf
global_mean = float(train_f.agg(F.avg("total_amount").alias("m")).first()["m"])
val_D  = (val_bins.join(leaves, on=[f"{c}_bin" for c in BFEATS], how="left")
               .withColumn("prediction_D", F.coalesce(F.col("leaf_pred"), F.lit(global_mean))))
test_D = (test_bins.join(leaves, on=[f"{c}_bin" for c in BFEATS], how="left")
               .withColumn("prediction_D", F.coalesce(F.col("leaf_pred"), F.lit(global_mean))))

# Score validation and test datasets:
rmse_val_D_true  = rmse(val_D,  "prediction_D", "total_amount")
rmse_test_D_true = rmse(test_D, "prediction_D", "total_amount")
rmse_val_D_robust = rmse(val_D,  "prediction_D", "label_robust")
rmse_test_D_robust = rmse(test_D,  "prediction_D", "label_robust")

print(f"Model D summary — TRUE:   val {rmse_val_D_true:.3f} | test {rmse_test_D_true:.3f}")
print(f"Model D summary — ROBUST: val {rmse_val_D_robust:.3f} | test {rmse_test_D_robust:.3f}")

### 3.8 Project Conclusion


In [0]:
# Compare model performance against the baseline
results = [
    ("Baseline",               rmse_val_baseline,   rmse_test_baseline),
    ("Model_A_ridge_closed",   rmse_val_A_true,     rmse_test_A_true),
    ("Model_B_ridge_gd",       rmse_val_B_true,     rmse_test_B_true),
    ("Model_C_huber",          rmse_val_C_true,     rmse_test_C_true),
    ("Model_D_tree_fallback",  rmse_val_D_true,     rmse_test_D_true),
]
best = sorted(results, key=lambda x: x[1])[0]
print("=== RMSE summary on TRUE labels (Val | Test) ===")
for name, rv, rt in results:
    print(f"{name:26s} {rv:8.3f} | {rt:8.3f}")
print(f"\nBest model by validation RMSE: {best[0]}")
print("\n(ROBUST RMSE for diagnostics)")
print(f"Model A: val {rmse_val_A_robust:.3f} | test {rmse_test_A_robust:.3f}")
print(f"Model D: val {rmse_val_D_robust :.3f} | test {rmse_test_D_robust :.3f}")


In [0]:
import itertools

# Z-scored feature columns
Z_FEATS = [
    "trip_distance_km_z","duration_min_z","passenger_count_z",
    "hour_z","month_z","dow_z","color_te_z","pu_te_z","do_te_z"
]

# --- Correlation of features with TRUE and ROBUST labels (train only)
def corr_with_label(df, features, label_col):
    rows = []
    for c in features:
        val = df.select(F.corr(F.col(c), F.col(label_col)).alias("r")).first()["r"]
        rows.append((c, float(val) if val is not None else None))
    return spark.createDataFrame(rows, ["feature", f"corr_{label_col}"])

corr_true   = corr_with_label(train_z, Z_FEATS, "total_amount")
corr_robust = corr_with_label(train_z, Z_FEATS, "label_robust")

corr_tbl = (
    corr_true.join(corr_robust, on="feature", how="inner")
             .withColumn("corr_total_amount",  F.round("corr_total_amount",  2))
             .withColumn("corr_label_robust",  F.round("corr_label_robust",  2))
             .orderBy(F.desc(F.abs("corr_total_amount")))
)
display(corr_tbl)  # appendix screenshot

# --- Standardized coefficients from Model A (trained on z's)
coef_rows = [(feat, float(w)) for feat, w in zip(FEAT_COLS, w_A)]
coef_df = (spark.createDataFrame(coef_rows, ["feature","coef"])
             .filter(F.col("feature") != "bias")
             .withColumn("abs_coef", F.abs("coef"))
             .withColumn("coef", F.format_number("coef", 2))        # format coef
             .withColumn("abs_coef", F.format_number("abs_coef", 2))  # format abs_coef
             .orderBy(F.desc("abs_coef")))
display(coef_df)   # appendix screenshot

# --- Approximate multicollinearity via VIF = diag(inv(R)) on z-features
# Build Pearson correlation matrix on TRAIN
corr = {}
for a, b in itertools.product(Z_FEATS, Z_FEATS):
    r = train_z.select(F.corr(F.col(a), F.col(b)).alias("r")).first()["r"]
    corr[(a,b)] = float(r) if r is not None else (1.0 if a == b else 0.0)

R = np.array([[corr[(a,b)] for b in Z_FEATS] for a in Z_FEATS], dtype=float)

# Invert with tiny ridge if needed
eps = 1e-6
try:
    Rinv = np.linalg.inv(R)
except np.linalg.LinAlgError:
    Rinv = np.linalg.inv(R + eps*np.eye(len(Z_FEATS)))

vif_vals = np.diag(Rinv)
vif_df = spark.createDataFrame(list(zip(Z_FEATS, [float(x) for x in vif_vals])),
                               ["feature_z","VIF"]).orderBy(F.desc("VIF"))
vif_df_fmt = vif_df.withColumn("VIF", F.format_number("VIF", 2))
display(vif_df_fmt) # appendix screenshot

In [0]:
# Persist Model A artifacts to the configured model directory
import datetime
import json

# --- Weights (feature order + coefficients)
weights_df = spark.createDataFrame(
    list(zip(FEAT_COLS, [float(x) for x in w_A.tolist()])),
    ["feature","weight"]
)
weights_df.write.mode("overwrite").format("delta").save(f"{MODEL_DIR}/weights")

# --- Z-score stats (mean/std per numeric/TE column used in standardization)
stats_rows = []
for k,v in stats.items():
    # k looks like "<col>_mean" or "<col>_std"
    parts = k.rsplit("_", 1)
    col_name, kind = parts[0], parts[1]
    stats_rows.append((col_name, kind, float(0.0 if v is None else v)))
schema_stats = T.StructType([
    T.StructField("col",  T.StringType(), False),
    T.StructField("stat", T.StringType(), False),
    T.StructField("value",T.DoubleType(), False),
])
spark.createDataFrame(stats_rows, schema_stats) \
     .write.mode("overwrite").format("delta").save(f"{MODEL_DIR}/zscore_stats")

# --- Target-encoding maps (train-only, smoothed)
for c in ["color", "pu_borough", "do_borough"]:
    te_df = te_maps[c]  # columns: c, f"{c}_te"
    te_df.write.mode("overwrite").format("delta").save(f"{MODEL_DIR}/te_{c}")

# --- Metadata JSON
meta = {
    "created_at": datetime.datetime.utcnow().isoformat() + "Z",
    "algo": "ridge_closed_form",
    "lambda": lam,
    "feature_order": FEAT_COLS,            # order used for weights
    "y_global_for_TE": float(y_global),    # for unseen categories
    "te_smoothing_m": 30.0,
    "robust_cap": float(CAP),              # train p99.95
    "standardized": True,
    "notes": "All stats/TE maps fit on TRAIN only; evaluation uses TRUE labels."
}
dbutils.fs.put(f"{MODEL_DIR}/metadata.json", json.dumps(meta, indent=2), overwrite=True)

print(f"Saved Model A artifacts under: {MODEL_DIR}")
display(dbutils.fs.ls(MODEL_DIR))